# Đánh giá test M0–M3

Tám lượt xuất, mỗi lượt 1.968 câu. Chạy từ trên xuống; chạy lại cùng launch record để nối phần còn thiếu.
Hướng dẫn chấm đã khóa trước test; phần luyện chấm và nhãn người được theo dõi riêng.
Notebook không train hoặc thay output lịch sử.


In [ ]:
import hashlib
import json
import os
import re
import runpy
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = os.environ.get('VQA_PROJECT_ROOT')
RESULTS_DIR = os.environ.get('VQA_TEST_RESULTS_DIR')
LAUNCH_RECORD = ''
LAUNCH_SHA256 = ''
RUN_TEST = False
RUN_IDENTITIES = None
RUN_DIR_NAMES = {}
RELOCATE_IDENTITIES = []
RELOCATION_SCRIPT = 'source/relocate_test_run.py'
RELOCATION_SHA256 = ''
INSPECTED = False


## 1. Kho dự án và source


In [ ]:
from google.colab import drive

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive', timeout_ms=600000)
if PROJECT_DIR:
    ROOT = Path(PROJECT_DIR).resolve()
else:
    candidates = [Path('/content/drive/MyDrive/Vietnamese_VQA_NLE')]
    candidates += list(Path('/content/drive/.shortcut-targets-by-id').glob('*/Vietnamese_VQA_NLE'))
    roots = {p.resolve() for p in candidates if (p / 'source/releases').is_dir()}
    if len(roots) != 1:
        raise ValueError('Set PROJECT_DIR to the mounted project folder')
    ROOT = roots.pop()
print('Project:', ROOT)


In [ ]:
INSPECTED = False
if not re.fullmatch(r'[0-9a-f]{64}', LAUNCH_SHA256):
    raise ValueError('Select the verified test launch record and SHA-256')

def project_path(relative):
    relative = Path(relative)
    folders = {'source', 'datasets', 'runs', 'evaluation_runs'}
    if (relative.is_absolute() or not relative.parts
            or relative.parts[0] not in folders or '..' in relative.parts):
        raise ValueError('Project path is outside the declared folders')
    folder = (ROOT / relative.parts[0]).resolve()
    path = (ROOT / relative).resolve()
    if not path.is_relative_to(folder):
        raise ValueError('Project path escapes its mounted folder')
    return path

launch_path = project_path(LAUNCH_RECORD)
if hashlib.sha256(launch_path.read_bytes()).hexdigest() != LAUNCH_SHA256:
    raise ValueError('Test launch changed')
launch = json.loads(launch_path.read_text('utf-8'))
if launch.get('schema') != 'final_test_launch_v1' or launch.get('questions_per_identity') != 1968:
    raise ValueError('Unexpected test launch record')
bootstrap = ROOT / 'source/bootstrap_cloud_source.py'
if hashlib.sha256(bootstrap.read_bytes()).hexdigest() != (
    '391172ed1fbf39d0eeeef2c9aa3d056d7770085d0a10574d571470d7f2bbef40'
):
    raise ValueError('Bootstrap checksum mismatch')

def restore(relative, digest, work_root):
    return Path(subprocess.check_output([
        sys.executable, str(bootstrap), '--archive', str(project_path(relative)),
        '--sha256', digest, '--work-root', work_root,
    ], text=True).strip())

REPO = restore(launch['source_archive'], launch['source_sha256'], '/content/vqa_sources')
DATA = restore(launch['inputs_archive'], launch['inputs_sha256'], '/content/vqa_test_inputs')
PLAN = DATA / 'frozen/plan.json'
if hashlib.sha256(PLAN.read_bytes()).hexdigest() != launch['plan_sha256']:
    raise ValueError('Frozen test plan changed')
plan = json.loads(PLAN.read_text('utf-8'))
OUTPUT = Path(RESULTS_DIR).resolve() if RESULTS_DIR else project_path(launch['output_dir'])
run_command = runpy.run_path(str(REPO / 'src/vqa_explanation/utils/process.py'))['run_command']

def run(command):
    run_command([str(part) for part in command], cwd=REPO)

print('Source:', REPO)
print('Output:', OUTPUT)
print('Human calibration:', 'pending')


## 2. Môi trường và kiểm đầu vào


In [ ]:
ENV_DIR = Path('/content/vqa_envs') / ('test_' + launch['source_sha256'][:12])
PYTHON = ENV_DIR / 'bin/python'
run([sys.executable, REPO / 'scripts/setup_cloud_environment.py',
     '--profile', 'preference', '--environment-dir', ENV_DIR])
ENVIRONMENT = ENV_DIR / 'test_environment.yaml'
environment = {'annotations': str(DATA / 'metadata/all.jsonl'), 'image_root': str(DATA),
               'run_root': str(ROOT / 'runs'),
               'image_checksums': str(DATA / 'metadata/full_image_checksums.csv')}
ENVIRONMENT.write_text(json.dumps(environment), encoding='utf-8')
run([PYTHON, '-c', 'import torch; assert torch.cuda.is_available(), "Select a GPU runtime"; '
     'print(torch.cuda.get_device_name(0))'])


In [ ]:
INSPECTED = False
COMMANDS = []
selected = set(plan['runs']) if RUN_IDENTITIES is None else set(RUN_IDENTITIES)
if not selected or not selected <= set(plan['runs']):
    raise ValueError('Select planned test identities')
if not set(RUN_DIR_NAMES) <= selected or not set(RELOCATE_IDENTITIES) <= selected:
    raise ValueError('Storage overrides must belong to selected identities')
for identity, record in plan['runs'].items():
    if identity not in selected:
        continue
    run_dir = OUTPUT / RUN_DIR_NAMES.get(identity, identity)
    arguments = ['--environment', ENVIRONMENT, '--plan', PLAN,
                 '--plan-sha256', launch['plan_sha256'], '--identity', identity,
                 '--output-dir', run_dir, '--rubric', DATA / 'frozen']
    if record['adapter_at_validation']:
        relative = record['adapter_at_validation'].split('/Vietnamese_VQA_NLE/', 1)[1]
        arguments += ['--adapter', project_path(relative)]
    if identity in RELOCATE_IDENTITIES:
        tool = project_path(RELOCATION_SCRIPT)
        if hashlib.sha256(tool.read_bytes()).hexdigest() != RELOCATION_SHA256:
            raise ValueError('Relocation tool checksum mismatch')
        run([PYTHON, tool, *arguments, '--source-dir', OUTPUT / identity])
    command = [PYTHON, '-u', REPO / 'scripts/run_study_test.py', *arguments]
    inspected = json.loads(subprocess.check_output(
        [str(part) for part in command + ['--inspect']], cwd=REPO, text=True))
    print(json.dumps(inspected, ensure_ascii=False, indent=2))
    if inspected['status'] != 'inputs_verified':
        raise RuntimeError(f"Test input blocked: {identity}")
    COMMANDS.append(command)
INSPECTED = True


## 3. Xuất và theo dõi

Output và các prefix phục hồi được lưu trên Drive trong lúc chạy.


In [ ]:
if not INSPECTED:
    raise RuntimeError('Complete input inspection first')
if RUN_TEST:
    for command in COMMANDS:
        run(command)
else:
    print('Test export not requested')


In [ ]:
progress_command = [PYTHON, REPO / 'scripts/report_test.py', '--plan', PLAN,
                    '--plan-sha256', launch['plan_sha256'],
                    '--annotations', DATA / 'metadata/test.jsonl',
                    '--runs-root', OUTPUT, '--inspect']
for identity, directory in RUN_DIR_NAMES.items():
    progress_command += ['--run', f'{identity}={OUTPUT / directory}']
run(progress_command)
